## GraphDB KC 데이터셋

### 1. 필요한 라이브러리

In [ ]:
import pandas as pd
from neo4j import GraphDatabase
import json

### 2. Neo4j 연결

In [ ]:
NEO4J_URI=""
NEO4J_USERNAME=""
NEO4J_PASSWORD=""

In [ ]:
driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USERNAME, NEO4J_PASSWORD))

def close_driver():
    driver.close()

print(driver.get_server_info())

### 3. 데이터 처리


In [ ]:
NODE_KEYWORD_DATA = ''
NODE_PAPER_DATA = ''
EDGE_KEYWORD_DATA = ''
EDGE_PAPER_DATA = ''

nodes_keyword_df = pd.read_csv(NODE_KEYWORD_DATA)
nodes_paper_df = pd.read_csv(NODE_PAPER_DATA)
edges_keyword_df = pd.read_csv(EDGE_KEYWORD_DATA)
edges_paper_df = pd.read_csv(EDGE_PAPER_DATA)

print(f"키워드 노드 수: {len(nodes_keyword_df)}")
print(f"논문 노드 수: {len(nodes_paper_df)}")
print(f"논문-키워드 엣지 수: {len(edges_keyword_df)}")
print(f"논문-논문 엣지 수: {len(edges_paper_df)}")

# 데이터 확인
print("\n=== 노드 샘플 ===")
print(nodes_keyword_df.head())
print(nodes_paper_df.head())

print("\n=== 엣지 샘플 ===")
print(edges_keyword_df.head())
print(edges_paper_df.head())

In [ ]:
import ast

# categories를 문자열에서 리스트로 변환
def parse_categories(cat_str):
    try:
        return ast.literal_eval(cat_str)
    except:
        return []

nodes_keyword_df['categories'] = nodes_keyword_df['categories'].apply(parse_categories)
nodes_paper_df['categories'] = nodes_paper_df['categories'].apply(parse_categories)

# alias 처리 (필요한 경우)
nodes_keyword_df['alias'] = nodes_keyword_df['alias'].fillna('')

print("데이터 전처리 완료")
print(nodes_keyword_df.head())
print(nodes_paper_df.head())

In [ ]:
import ast

# categories를 문자열에서 리스트로 변환
def parse_edges(list_str):
    try:
        return ast.literal_eval(list_str)
    except:
        return []
    
edges_paper_df['intents'] = edges_paper_df['intents'].apply(parse_edges)
edges_paper_df['contexts'] = edges_paper_df['contexts'].apply(parse_edges)

edges_paper_df.head()

### 4. Keyword Node 생성

In [ ]:
def create_constraints(tx):
    # CONSTRAINT
    try:
        tx.run("CREATE CONSTRAINT keyword_name_unique IF NOT EXISTS FOR (k:Keyword) REQUIRE k.name IS UNIQUE")
        print("Uniqueness constraint 생성 완료")
    except Exception as e:
        print(f"Constraint 생성 중 오류 (이미 존재할 수 있음): {e}")

def create_indexes(tx):
    # INDEX
    try:
        tx.run("CREATE INDEX keyword_name_index IF NOT EXISTS FOR (k:Keyword) ON (k.name)")
        print("Index 생성 완료")
    except Exception as e:
        print(f"Index 생성 중 오류: {e}")

with driver.session() as session:
    session.execute_write(create_constraints)
    session.execute_write(create_indexes)

In [ ]:
def create_keyword_node(tx, name, link, alias, categories):
    query = """
    MERGE (k:Keyword {name: node.name})
    SET k.link = node.link,
        k.alias = node.alias,
        k.categories = node.categories
    """
    result = tx.run(query, name=name, link=link, alias=alias, categories=categories)
    return result.single()['node_id']

def create_nodes_batch(tx, nodes_batch):
    query = """
    UNWIND $nodes as node
    MERGE (k:Keyword {name: node.name})
    SET k.link = node.link,
        k.alias = node.alias,
        k.categories = node.categories
    """
    tx.run(query, nodes=nodes_batch)

In [ ]:
batch_size = 1000
total_nodes = len(nodes_keyword_df)

with driver.session() as session:
    for i in range(0, total_nodes, batch_size):
        batch = nodes_keyword_df.iloc[i:i+batch_size]
        nodes_batch = [
            {
                'name': row['name'],
                'link': row['link'],
                'alias': row['alias'],
                'categories': row['categories']
            }
            for _, row in batch.iterrows()
        ]
        session.execute_write(create_nodes_batch, nodes_batch)
        print(f"노드 생성 진행: {min(i+batch_size, total_nodes)}/{total_nodes}")

print(f"\n총 생성 노드: {total_nodes}")

### 5. Paper Node 생성

In [ ]:
def create_paper_constraints(tx):
    # CONSTRAINT
    try:
        tx.run("CREATE CONSTRAINT paper_name_unique IF NOT EXISTS FOR (p:Paper) REQUIRE p.name IS UNIQUE")
        print("Uniqueness constraint 생성 완료")
    except Exception as e:
        print(f"Constraint 생성 중 오류 (이미 존재할 수 있음): {e}")

def create_paper_indexes(tx):
    # INDEX
    try:
        tx.run("CREATE INDEX paper_name_index IF NOT EXISTS FOR (p:Paper) ON (p.name)")
        tx.run("CREATE INDEX paper_paperId_index IF NOT EXISTS FOR (p:Paper) ON (p.paperId)")
        print("Index 생성 완료")
    except Exception as e:
        print(f"Index 생성 중 오류: {e}")

with driver.session() as session:
    session.execute_write(create_paper_constraints)
    session.execute_write(create_paper_indexes)

In [ ]:
def create_paper_node(tx, name, paper_id, categories, year, url, authors, abstract, publication, reference_count, citation_count):
    query = """
    MERGE (p:Paper {name: node.name})
    SET p.paperId = node.paperId,
        p.categories = node.categories,
        p.year = node.year,
        p.url = node.url,
        p.authors = node.authors,
        p.abstract = node.abstract,
        p.publication = node.publication,
        p.referenceCount = node.referenceCount,
        p.citationCount = node.citationCount
    """
    result = tx.run(query, name=name, paperId=paper_id, categories=categories, year=year, url=url, authors=authors, abstract=abstract, publication=publication, reference_count=reference_count, citation_count=citation_count)
    return result.single()[['node_id']]


def create_paper_node_batch(tx, nodes_batch):
    query = """
    UNWIND $nodes as node
    MERGE (p:Paper {name: node.name})
    SET p.paperId = node.paperId,
        p.categories = node.categories,
        p.year = node.year,
        p.url = node.url,
        p.authors = node.authors,
        p.abstract = node.abstract,
        p.publication = node.publication,
        p.referenceCount = node.referenceCount,
        p.citationCount = node.citationCount
    """
    tx.run(query, nodes=nodes_batch)

In [ ]:
batch_size = 1000
total_nodes = len(nodes_paper_df)

with driver.session() as session:
    for i in range(0, total_nodes, batch_size):
        batch = nodes_paper_df.iloc[i:i+batch_size]
        nodes_batch = [
            {
                'paperId': row['paperId'],
                'year': row['year'],
                'url': row['url'],
                'name': row['title'],
                'authors': row['authors'],
                'abstract': row['abstract'],
                'publication': row['publication'],
                'referenceCount': row['referenceCount'],
                'citationCount': row['citationCount']
            }
            for _, row in batch.iterrows()
        ]
        session.execute_write(create_paper_node_batch, nodes_batch)
        print(f"노드 생성 진행: {min(i+batch_size, total_nodes)}/{total_nodes}")


### 5. Keyword-Paper Edge 생성

In [ ]:
def create_about_relation_batch(tx, edges_batch):
    query = """
    UNWIND $edges as edge
    MATCH (source:Paper {paperId: edge.source_id})
    MATCH (target:Keyword {name: edge.target_name})
    MERGE (source)-[r:ABOUT]->(target)
    SET r.strength = edge.strength,
        r.reason = edge.reason
    """
    tx.run(query, edges=edges_batch)


def create_in_relation_batch(tx, edges_batch):
    query = """
    UNWIND $edges as edge
    MATCH (source:Keyword {name: edge.source_name})
    MATCH (target:Paper {paperId: edge.target_id})
    MERGE (source)-[r:IN]->(target)
    SET r.strength = edge.strength,
        r.reason = edge.reason
    """
    tx.run(query, edges=edges_batch)

In [ ]:
edge_about_df = edges_keyword_df[edges_keyword_df['name'] == "ABOUT"]

batch_size = 1000
total_edges = len(edge_about_df)

with driver.session() as session:
    for i in range(0, total_edges, batch_size):
        batch = edge_about_df.iloc[i:i+batch_size]
        edges_batch = [
            {
                'source_id': row['paperId'],
                'target_name': row['concept'],
                'strength': float(row['strength']),
                'reason': row['reason'] if pd.notna(row['reason']) else None
            }
            for _, row in batch.iterrows()
        ]
        session.execute_write(create_about_relation_batch, edges_batch)
        print(f"엣지 생성 진행: {min(i+batch_size, total_edges)}/{total_edges}")

print(f"\n총 생성 엣지: {total_edges}")

In [ ]:
edge_in_df = edges_keyword_df[edges_keyword_df['name'] == "IN"]

batch_size = 1000
total_edges = len(edge_in_df)

with driver.session() as session:
    for i in range(0, total_edges, batch_size):
        batch = edge_in_df.iloc[i:i+batch_size]
        edges_batch = [
            {
                'source_name': row['concept'],
                'target_id': row['paperId'],
                'strength': float(row['strength']),
                'reason': row['reason'] if pd.notna(row['reason']) else None
            }
            for _, row in batch.iterrows()
        ]
        session.execute_write(create_in_relation_batch, edges_batch)
        print(f"엣지 생성 진행: {min(i+batch_size, total_edges)}/{total_edges}")

print(f"\n총 생성 엣지: {total_edges}")

### 6. Paper-Paper Edge 생성

In [ ]:
def create_ref_relation_batch(tx, edges_batch):
    query = """
    UNWIND $edges as edge
    MATCH (source:Paper {paperId: edge.source_id})
    MATCH (target:Paper {paperId: edge.target_id})
    MERGE (source)-[r:REF_BY]->(target)
    SET r.intents = edge.intents,
        r.isInfluential = edge.isInfluential,
        r.strength = edge.strength,
        r.contexts = edge.contexts
    """
    tx.run(query, edges=edges_batch)

In [ ]:
batch_size = 1000
total_edges = len(edges_paper_df)

with driver.session() as session:
    for i in range(0, total_edges, batch_size):
        batch = edges_paper_df.iloc[i:i+batch_size]
        edges_batch = [
            {
                'source_id': row['ref_paper_id'],
                'target_id': row['seed_paper_id'],
                'intents': row['intents'],
                'isInfluential': row['isInfluential'],
                'contexts': row['contexts'],
                'strength': row['strength']
            }
            for _, row in batch.iterrows()
        ]
        session.execute_write(create_ref_relation_batch, edges_batch)
        print(f"엣지 생성 진행: {min(i+batch_size, total_edges)}/{total_edges}")

print(f"\n총 생성 엣지: {total_edges}")

### 7. 결과 확인 코드

In [ ]:
def get_graph_stats(tx):
    # 노드 수
    node_count = tx.run("MATCH (k:Keyword) RETURN count(k) as count").single()['count']
    
    # 엣지 수
    edge_count = tx.run("MATCH ()-[r:PREREQ]->() RETURN count(r) as count").single()['count']
    
    return node_count, edge_count

def get_sample_nodes(tx, limit=5):
    result = tx.run("""
        MATCH (k:Keyword)
        RETURN k.name as name, k.link as link, k.categories as categories
        LIMIT $limit
    """, limit=limit)
    return [record.data() for record in result]

def get_sample_relationships(tx, limit=5):
    result = tx.run("""
        MATCH (source:Keyword)-[r:PREREQ]->(target:Keyword)
        RETURN source.name as source, target.name as target, 
               r.strength as strength, r.reason as reason
        LIMIT $limit
    """, limit=limit)
    return [record.data() for record in result]

# 통계 확인
with driver.session() as session:
    node_count, edge_count = session.execute_read(get_graph_stats)
    print(f"=== 그래프 통계 ===")
    print(f"총 노드 수: {node_count}")
    print(f"총 엣지 수: {edge_count}")
    
    print(f"\n=== 샘플 노드 (5개) ===")
    sample_nodes = session.execute_read(get_sample_nodes)
    for node in sample_nodes:
        print(f"- {node['name']}: {node['link']}")
        print(f"  Categories: {node['categories']}")
    
    print(f"\n=== 샘플 관계 (5개) ===")
    sample_rels = session.execute_read(get_sample_relationships)
    for rel in sample_rels:
        print(f"- {rel['source']} -[PREREQ]-> {rel['target']}")
        print(f"  Strength: {rel['strength']}, Reason: {rel['reason']}")
